In [ ]:
from datetime import datetime as dt

track_data = {
    "datetime": [
        "2014-09-10 18:00", "2014-09-11 00:00", "2014-09-11 06:00", "2014-09-11 12:00",
        "2014-09-11 18:00", "2014-09-12 00:00", "2014-09-12 06:00", "2014-09-12 12:00",
        "2014-09-12 18:00", "2014-09-13 00:00", "2014-09-13 06:00", "2014-09-13 12:00",
        "2014-09-13 18:00", "2014-09-14 00:00", "2014-09-14 06:00", "2014-09-14 12:00",
        "2014-09-14 18:00", "2014-09-15 00:00", "2014-09-15 06:00", "2014-09-15 12:00",
        "2014-09-15 18:00", "2014-09-16 00:00", "2014-09-16 06:00", "2014-09-16 12:00",
        "2014-09-16 18:00", "2014-09-17 00:00", "2014-09-17 06:00", "2014-09-17 12:00",
        "2014-09-17 18:00", "2014-09-18 00:00", "2014-09-18 06:00", "2014-09-18 12:00",
        "2014-09-18 18:00", "2014-09-19 00:00", "2014-09-19 06:00", "2014-09-19 12:00",
        "2014-09-19 18:00", "2014-09-20 00:00", "2014-09-20 06:00", "2014-09-20 12:00",
        "2014-09-20 18:00"
    ],
    "lat": [
        13.7, 14.2, 14.9, 15.8, 16.5, 17.2, 17.7, 18.3, 18.9, 19.6, 20.4, 21.4, 22.3,
        23.0, 23.7, 24.4, 25.1, 25.8, 26.5, 27.0, 27.7, 28.5, 29.5, 30.6, 31.7,
        32.9, 34.3, 35.7, 37.2, 38.3, 39.3, 39.9, 39.9, 39.9, 39.9, 39.5, 39.5,
        39.4, 39.3, 39.1, 38.8
    ],
    "lon": [
        -33.2, -34.4, -35.5, -36.5, -37.7, -39.2, -40.8, -42.3, -43.6, -44.8,
        -45.7, -46.4, -47.0, -47.9, -49.0, -50.3, -51.5, -52.7, -54.0, -55.1,
        -56.1, -56.7, -57.3, -57.8, -57.7, -57.0, -56.0, -54.5, -52.4, -49.5,
        -46.7, -44.1, -42.1, -40.1, -38.9, -38.1, -37.4, -36.6, -35.7, -34.4, -33.0
    ],
    "pressure": [
        1009, 1009, 1009, 1008, 1007, 1006, 1005, 1000, 999, 998, 997, 996, 995,
        994, 991, 985, 980, 975, 970, 966, 962, 960, 956, 955, 958, 959, 958,
        957, 955, 958, 964, 969, 973, 980, 988, 994, 1001, 1004, 1006, 1008, 1010
    ],
    "wind": [
        25, 25, 25, 30, 30, 35, 35, 40, 40, 40, 45, 45, 50, 55, 60, 70, 75, 80,
        85, 90, 95, 95, 100, 105, 95, 85, 85, 80, 80, 75, 70, 70, 65, 60, 55,
        45, 40, 35, 30, 30, 25
    ],
    "stage": [
        "low", "low", "low", "tropical depression", "tropical depression",
        "tropical storm", "tropical storm", "tropical storm", "tropical storm",
        "tropical storm", "tropical storm", "tropical storm", "tropical storm",
        "tropical storm", "tropical storm", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "hurricane", "hurricane", "hurricane", "hurricane", "hurricane",
        "tropical storm", "tropical storm", "tropical storm", "low", "low",
        "low", "low", "low"
    ]
}


In [ ]:
import os, glob, subprocess, datetime as dt
import pandas as pd, numpy as np, xarray as xr, matplotlib.pyplot as plt
from matplotlib.patches import Circle
from tropycal import tracks

# --- Time range ---
start_cutoff, end_cutoff = pd.Timestamp("2014-09-11"), pd.Timestamp("2014-09-18 23:59")

# --- Track data ---
df_track = pd.DataFrame(track_data).assign(datetime=lambda x: pd.to_datetime(x["datetime"])).set_index("datetime")
interp_times = pd.date_range(df_track.index.min(), df_track.index.max(), freq="30min")
track_interp = df_track.reindex(df_track.index.union(interp_times))
track_interp = track_interp.interpolate("time").loc[start_cutoff:end_cutoff]
track_interp["stage"] = df_track["stage"].reindex(track_interp.index).ffill()

# --- Output folder ---
os.makedirs("outputs_edouard_frames", exist_ok=True)

# --- Datasets ---
ds_merg = xr.open_mfdataset(sorted(glob.glob("/spare11/atm350/es348865/MERGIR_data/merg_201409*.nc4")), combine="by_coords")
df_light_all = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("/nfs/home11/ugrad/2024/es348865/honors_research/250523/WWLLN_201409*.csv"))])
df_light_all["longitude"] = ((df_light_all["longitude"] + 180) % 360) - 180
df_light_all["datetime"] = pd.to_datetime(df_light_all["datetime"], format="%Y/%m/%d %H:%M:%S.%f")

# --- SHIPS shear ---
storm = tracks.TrackDataset(basin="north_atlantic").get_storm(("edouard", 2014))
ships = storm.get_ships(dt.datetime(2014, 9, 11, 12))
times = [ships.attrs["forecast_init"] + dt.timedelta(hours=int(h)) for h in ships.fhr]
df_ships = pd.DataFrame({
    "datetime": times,
    "u_shear": -ships.shear_kt * np.sin(np.deg2rad(ships.shear_dir)),
    "v_shear": -ships.shear_kt * np.cos(np.deg2rad(ships.shear_dir)),
    "shear_mag": ships.shear_kt
}).set_index("datetime")

# --- Interpolated shear ---
df_ships_interp = pd.DataFrame(index=interp_times)
for col in ["u_shear", "v_shear", "shear_mag"]:
    df_ships_interp[col] = np.interp(interp_times.view(int), df_ships.index.view(int), df_ships[col])


In [ ]:
ds_merg

In [ ]:
pip install tqdm


In [ ]:
# --- FIG.5-style TB vs. shear-relative azimuth (smooth, quadrant-labeled) ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from matplotlib.dates import DateFormatter
import matplotlib.dates as mdates
from tqdm import tqdm

# ---------------- Parameters ----------------
inner_radius_deg = 2.0          # inner 1° radial average
azimuth_res_deg = 1.0           # azimuth bin width
fixed_shear_heading = 8.0       # degrees
time_start = pd.Timestamp("2014-09-11 12:00")
time_end   = pd.Timestamp("2014-09-12 12:00")  # 24h window

# ---------------- Select times ----------------
times = track_interp.index[(track_interp.index >= time_start) & (track_interp.index < time_end)]
if len(times) == 0:
    raise ValueError("No times in track_interp within requested range.")
print(f"Processing {len(times)} time steps for test run.")

# ---------------- Azimuth bins ----------------
az_edges = np.arange(0, 360 + azimuth_res_deg, azimuth_res_deg)
az_centers = (az_edges[:-1] + az_edges[1:]) / 2.0

# ---------------- Restrict spatial domain ----------------
lat0, lon0 = float(track_interp.loc[times[0], "lat"]), float(track_interp.loc[times[0], "lon"])
lat_mask = (ds_merg["lat"] >= lat0 - 3) & (ds_merg["lat"] <= lat0 + 3)
lon_mask = (ds_merg["lon"] >= lon0 - 3) & (ds_merg["lon"] <= lon0 + 3)
ds_merg_small = ds_merg.sel(lat=lat_mask, lon=lon_mask)

# ---------------- Precompute azimuth/radial grids ----------------
lons, lats = ds_merg_small["lon"].values, ds_merg_small["lat"].values
lon2d, lat2d = np.meshgrid(lons, lats)

az_all, r_all = {}, {}
for storm_time in tqdm(times, desc="Precomputing az/r grids"):
    lat, lon = float(track_interp.loc[storm_time, "lat"]), float(track_interp.loc[storm_time, "lon"])
    lat_rad = np.deg2rad(lat)
    dx = (lon2d - lon) * np.cos(lat_rad)
    dy = (lat2d - lat)
    az_all[storm_time] = (np.degrees(np.arctan2(dx, dy)) + 360) % 360
    r_all[storm_time] = np.sqrt(dx**2 + dy**2)

# ---------------- Container for TB means ----------------
tb_time_az = np.full((len(times), len(az_centers)), np.nan)

# ---------------- Main loop ----------------
for ti, storm_time in enumerate(tqdm(times, desc="Computing TB azimuth means")):
    tb = ds_merg_small["Tb"].sel(time=storm_time, method="nearest").values

    lat, lon = float(track_interp.loc[storm_time, "lat"]), float(track_interp.loc[storm_time, "lon"])
    lat_rad = np.deg2rad(lat)
    dx = (lon2d - lon) * np.cos(lat_rad)
    dy = (lat2d - lat)
    az_grid = (np.degrees(np.arctan2(dx, dy)) + 360) % 360
    r_grid = np.sqrt(dx**2 + dy**2)

    mask_inner = r_grid <= inner_radius_deg
    rel_az = (az_grid - fixed_shear_heading) % 360
    tb_vals = tb[mask_inner]

    # Bin by azimuth
    az_bin_idx = np.digitize(rel_az[mask_inner], az_edges) - 1
    for ai in range(len(az_centers)):
        vals = tb_vals[az_bin_idx == ai]
        if vals.size > 0:
            tb_time_az[ti, ai] = np.nanmean(vals)

# ---------------- Prepare time and azimuth edges ----------------
time_array = np.array(times)
time_edges = np.append(time_array, time_array[-1] + pd.Timedelta(minutes=30))
time_edges_num = mdates.date2num(pd.to_datetime(time_edges))  # numeric dates for plotting

# ---------------- FIGURE ----------------
fig, ax = plt.subplots(figsize=(11, 5))

pcm = ax.pcolormesh(
    time_edges_num,
    az_edges,
    tb_time_az.T,
    cmap="turbo",
    vmin=180, vmax=300,
    shading="auto"
)

cbar = fig.colorbar(pcm, ax=ax, pad=0.02, label="Brightness Temperature (K)")

# ---------------- Axes formatting ----------------
ax.set_ylabel("Shear-relative azimuth (°)\n(0° = down-shear)", labelpad=1)
ax.set_ylim(0, 360)
ax.set_xlim(time_start, time_end)
ax.set_title(f"Inner-1° TB vs. Shear-Relative Azimuth\n{time_start:%Y-%m-%d %H:%M} – {time_end:%Y-%m-%d %H:%M}", y=1.15)
ax.xaxis_date()
ax.xaxis.set_major_formatter(DateFormatter('%m-%d\n%H:%M'))

# ---------------- Daily separators ----------------
for day in pd.date_range(time_start.normalize(), time_end.normalize() + pd.Timedelta(days=1), freq="D"):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)

# ---------------- Shear quadrants (label only, continuous data) ----------------
quad_centers = [45, 135, 225, 315]  # approx. center of each quadrant
quad_labels = ["DSR", "DSL", "USL", "USR"]
ax.set_yticks(quad_centers)
ax.set_yticklabels(quad_labels)

# ---------------- Shear magnitude inset ----------------
shear_mag_series = df_ships_interp.loc[time_array, "shear_mag"].reindex(time_array, fill_value=np.nan).values
ax_inset = fig.add_axes([0.1, .95, 0.7, 0.05], sharex=ax)
ax_inset.plot(time_array, shear_mag_series, lw=1.2)
ax_inset.set_ylabel("Shear (kt)", fontsize=8)
ax_inset.grid(True, ls=":", lw=0.5)
ax_inset.xaxis.set_visible(False)
ax_inset.set_ylim(np.nanmin(shear_mag_series) - 2, np.nanmax(shear_mag_series) + 2)

plt.subplots_adjust(bottom=0.12, top=0.9, left=0.1, right=0.95, hspace=0.3)
plt.show()


In [ ]:
# --- FIG. 5-style TB vs. shear-relative azimuth time series (full week, detailed comments) ---


########################            3 degrees from center, fetched every 30 minutes               ###########################
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from matplotlib.dates import DateFormatter
import matplotlib.dates as mdates
from tqdm import tqdm

# ---------------- PARAMETERS ----------------
inner_radius_deg = 3.0          # Radial distance (degrees) around the storm center to average TB
azimuth_res_deg = 1.0           # Azimuthal bin size (degrees)
fixed_shear_heading = 8.0       # Shear heading (°) relative to which azimuth is defined
time_start = pd.Timestamp("2014-09-11 12:00")
time_end   = pd.Timestamp("2014-09-18 12:00")  # Full week to plot

# ---------------- SELECT TIMES ----------------
# Only include track times within the requested time window
times = track_interp.index[(track_interp.index >= time_start) & (track_interp.index <= time_end)]
if len(times) == 0:
    raise ValueError("No times in track_interp within requested range.")
print(f"Processing {len(times)} time steps for full week.")

# ---------------- AZIMUTH BINS ----------------
# Define edges and centers of azimuth bins for averaging TB
az_edges = np.arange(0, 360 + azimuth_res_deg, azimuth_res_deg)
az_centers = (az_edges[:-1] + az_edges[1:]) / 2.0

# ---------------- SPATIAL SUBSET ----------------
# Restrict the TB dataset to a region covering the full track ±5° to improve computation speed
lat_min, lat_max = track_interp["lat"].min() - 5, track_interp["lat"].max() + 5
lon_min, lon_max = track_interp["lon"].min() - 5, track_interp["lon"].max() + 5
ds_merg_small = ds_merg.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

# ---------------- PRECOMPUTE AZIMUTH/RADIAL GRIDS ----------------
# For each storm time, calculate the azimuth (degrees from north) and radial distance (deg) for every grid point
lons, lats = ds_merg_small["lon"].values, ds_merg_small["lat"].values
lon2d, lat2d = np.meshgrid(lons, lats)

az_all, r_all = {}, {}
for storm_time in tqdm(times, desc="Precomputing az/r grids"):
    lat, lon = float(track_interp.loc[storm_time, "lat"]), float(track_interp.loc[storm_time, "lon"])
    lat_rad = np.deg2rad(lat)
    dx = (lon2d - lon) * np.cos(lat_rad)  # longitude scaled by latitude
    dy = (lat2d - lat)                     # latitude difference
    az_all[storm_time] = (np.degrees(np.arctan2(dx, dy)) + 360) % 360  # azimuth in degrees
    r_all[storm_time] = np.sqrt(dx**2 + dy**2)                          # radial distance

# ---------------- CONTAINER FOR TB MEANS ----------------
# Will store TB averaged over inner_radius for each azimuth bin at each time
tb_time_az = np.full((len(times), len(az_centers)), np.nan)

# ---------------- MAIN LOOP: COMPUTE AZIMUTHAL TB MEANS ----------------
for ti, storm_time in enumerate(tqdm(times, desc="Computing TB azimuth means")):
    tb = ds_merg_small["Tb"].sel(time=storm_time, method="nearest").values  # grab TB at nearest time

    az_grid = az_all[storm_time]
    r_grid = r_all[storm_time]

    # Only include points within the inner radius
    mask_inner = r_grid <= inner_radius_deg
    rel_az = (az_grid - fixed_shear_heading) % 360  # shear-relative azimuth
    tb_vals = tb[mask_inner]

    # Bin TB values by azimuth and compute mean
    az_bin_idx = np.digitize(rel_az[mask_inner], az_edges) - 1
    for ai in range(len(az_centers)):
        vals = tb_vals[az_bin_idx == ai]
        if vals.size > 0:
            tb_time_az[ti, ai] = np.nanmean(vals)

# ---------------- PREPARE TIME EDGES FOR PLOTTING ----------------
# Pcolormesh requires edges (one more than centers)
time_array = np.array(times)
time_edges = pd.date_range(start=time_start, end=time_end + pd.Timedelta(minutes=30), freq="30min")
time_edges_num = mdates.date2num(time_edges)  # convert to matplotlib numeric date format

# ---------------- FIGURE ----------------
fig, ax = plt.subplots(figsize=(12, 5))

# Plot TB as a function of time (x) and shear-relative azimuth (y)
pcm = ax.pcolormesh(
    time_edges_num,
    az_edges,
    tb_time_az.T,
    cmap="turbo",
    vmin=180, vmax=300,
    shading="auto"
)

# Colorbar
cbar = fig.colorbar(pcm, ax=ax, pad=0.02, label="Brightness Temperature (K)")

# ---------------- AXES FORMATTING ----------------
ax.set_ylabel("Shear-relative azimuth (°)\n(0° = down-shear)", labelpad=1)
ax.set_ylim(0, 360)
ax.set_yticks(np.arange(0, 361, 45))
ax.set_xlim(time_start, time_end)
ax.set_title(f"Inner-3° TB vs. Shear-Relative Azimuth\n{time_start:%Y-%m-%d %H:%M} – {time_end:%Y-%m-%d %H:%M}", y=1.15)
ax.xaxis_date()
ax.xaxis.set_major_formatter(DateFormatter('%m-%d\n%H:%M'))

# Daily vertical separators
for day in pd.date_range(time_start.normalize(), time_end.normalize() + pd.Timedelta(days=1), freq="D"):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)

# ---------------- SHEAR QUADRANTS (DSR → DSL → USL → USR) ----------------
quad_ranges = {"DSR": (0, 90), "DSL": (90, 180), "USL": (180, 270), "USR": (270, 360)}
for label, (a0, a1) in quad_ranges.items():
    center_ang = (a0 + a1) / 2.0
    ax.text(time_edges_num[0] - (time_edges_num[-1] - time_edges_num[0]) * 0.01,
            center_ang, label, ha="right", va="center", fontsize=9,
            fontweight="bold", bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"))

# ---------------- SHEAR MAGNITUDE INSET ----------------
shear_mag_series = df_ships_interp.loc[time_array, "shear_mag"].reindex(time_array, fill_value=np.nan).values
ax_inset = fig.add_axes([0.1, 0.95, 0.7, 0.05], sharex=ax)
ax_inset.plot(time_array, shear_mag_series, lw=1.2)
ax_inset.set_ylabel("Shear (kt)", fontsize=8)
ax_inset.grid(True, ls=":", lw=0.5)
ax_inset.xaxis.set_visible(False)
ax_inset.set_ylim(np.nanmin(shear_mag_series) - 2, np.nanmax(shear_mag_series) + 2)

plt.subplots_adjust(bottom=0.12, top=0.9, left=0.1, right=0.95, hspace=0.3)
plt.show()


With Lightning Data:

In [ ]:
print(df_track.columns)


In [ ]:
import pandas as pd

# Create DataFrame
df_track = pd.DataFrame(track_data)

# Convert datetime strings to pandas Timestamps
df_track["datetime"] = pd.to_datetime(df_track["datetime"], format="%Y-%m-%d %H:%M")

print(df_track.head())


In [ ]:
# --- FIG. 5-style TB vs. shear-relative azimuth time series (full week, detailed, monotonic edges) ---

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from matplotlib.dates import DateFormatter
import matplotlib.dates as mdates
from tqdm import tqdm

# ---------------- PARAMETERS ----------------
inner_radius_deg = 3.0          # radial distance (degrees) around storm center to average TB
azimuth_res_deg  = 1.0          # azimuthal bin size in degrees
fixed_shear_heading = 8.0       # shear heading relative to which azimuth is defined
time_start = pd.Timestamp("2014-09-11 12:00")
time_end   = pd.Timestamp("2014-09-18 12:00")  # full week

# ---------------- SELECT TIMES ----------------
times = track_interp.index[(track_interp.index >= time_start) & (track_interp.index <= time_end)]
if len(times) == 0:
    raise ValueError("No times in track_interp within requested range.")
print(f"Processing {len(times)} time steps for full week.")

# ---------------- AZIMUTH BINS ----------------
az_edges = np.arange(0, 360 + azimuth_res_deg, azimuth_res_deg)
az_centers = (az_edges[:-1] + az_edges[1:]) / 2.0

# ---------------- SPATIAL SUBSET ----------------
lat_min, lat_max = track_interp["lat"].min() - 5, track_interp["lat"].max() + 5
lon_min, lon_max = track_interp["lon"].min() - 5, track_interp["lon"].max() + 5
ds_merg_small = ds_merg.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

# ---------------- PRECOMPUTE AZIMUTH/RADIAL GRIDS ----------------
lons, lats = ds_merg_small["lon"].values, ds_merg_small["lat"].values
lon2d, lat2d = np.meshgrid(lons, lats)

az_all, r_all = {}, {}
for storm_time in tqdm(times, desc="Precomputing az/r grids"):
    lat, lon = float(track_interp.loc[storm_time, "lat"]), float(track_interp.loc[storm_time, "lon"])
    lat_rad = np.deg2rad(lat)
    dx = (lon2d - lon) * np.cos(lat_rad)
    dy = (lat2d - lat)
    az_all[storm_time] = (np.degrees(np.arctan2(dx, dy)) + 360) % 360
    r_all[storm_time] = np.sqrt(dx**2 + dy**2)

# ---------------- CONTAINER FOR TB ----------------
tb_time_az = np.full((len(times), len(az_centers)), np.nan)

# ---------------- COMPUTE AZIMUTHAL TB MEANS ----------------
for ti, storm_time in enumerate(tqdm(times, desc="Computing TB azimuth means")):
    tb = ds_merg_small["Tb"].sel(time=storm_time, method="nearest").values
    az_grid = az_all[storm_time]
    r_grid  = r_all[storm_time]

    # Include points within inner radius
    mask_inner = r_grid <= inner_radius_deg
    rel_az = (az_grid - fixed_shear_heading) % 360
    tb_vals = tb[mask_inner]

    # Bin TB by azimuth
    az_bin_idx = np.digitize(rel_az[mask_inner], az_edges) - 1
    for ai in range(len(az_centers)):
        vals = tb_vals[az_bin_idx == ai]
        if vals.size > 0:
            tb_time_az[ti, ai] = np.nanmean(vals)

# ---------------- ROTATE AZIMUTH SO 90° IS AT TOP ----------------
shift_bins = int(90 / azimuth_res_deg)
tb_time_az_rot = np.roll(tb_time_az, -shift_bins, axis=1)

# Create monotonic edges starting at 90 → 450 (no wrap)
az_edges_rot = np.arange(90, 90 + 360 + azimuth_res_deg, azimuth_res_deg)
az_centers_rot = (az_edges_rot[:-1] + az_edges_rot[1:]) / 2.0

# ---------------- TIME EDGES FOR PLOTTING ----------------
time_array = np.array(times)
time_edges = pd.date_range(start=time_start, end=time_end + pd.Timedelta(minutes=30), freq="30min")
time_edges_num = mdates.date2num(time_edges)

# ---------------- PLOTTING ----------------
fig, ax = plt.subplots(figsize=(12, 5))

# TB vs time and shear-relative azimuth
pcm = ax.pcolormesh(
    time_edges_num,
    az_edges_rot,
    tb_time_az_rot.T,
    cmap="turbo",
    vmin=180,
    vmax=300,
    shading="auto"
)

# Colorbar
cbar = fig.colorbar(pcm, ax=ax, pad=0.02, label="Brightness Temperature (K)")
cbar.ax.invert_yaxis()  # optional: invert y to match typical plots

# ---------------- AXES FORMATTING ----------------
ax.set_ylabel("Shear-relative azimuth (°)\n(0° = down-shear)", labelpad=1)
ax.set_ylim(az_edges_rot[0], az_edges_rot[-1])

# Tick positions every 45°, labeled 0..359
yticks_pos = np.arange(90, 90 + 360, 45)
yticks_labels = (yticks_pos % 360).astype(int)
ax.set_yticks(yticks_pos)
ax.set_yticklabels(yticks_labels)

ax.set_xlim(time_start, time_end)
ax.set_title(
    f"Inner-3° TB vs. Shear-Relative Azimuth\n{time_start:%Y-%m-%d %H:%M} – {time_end:%Y-%m-%d %H:%M}",
    y=1.15
)
ax.xaxis_date()
ax.xaxis.set_major_formatter(DateFormatter('%m-%d\n%H:%M'))

# Daily separators
for day in pd.date_range(time_start.normalize(), time_end.normalize() + pd.Timedelta(days=1), freq="D"):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)

# ---------------- SHEAR QUADRANT LABELS ----------------
quad_ranges = {"DSR": (0, 90), "DSL": (90, 180), "USL": (180, 270), "USR": (270, 360)}
for label, (a0, a1) in quad_ranges.items():
    center_ang = (a0 + a1) / 2.0
    ax.text(
        time_edges_num[0] - (time_edges_num[-1] - time_edges_num[0]) * 0.01,
        center_ang,
        label,
        ha="right",
        va="center",
        fontsize=9,
        fontweight="bold",
        bbox=dict(facecolor="white", alpha=0.7, edgecolor="none")
    )

# ---------------- SHEAR MAGNITUDE INSET ----------------
shear_mag_series = df_ships_interp.loc[time_array, "shear_mag"].reindex(time_array, fill_value=np.nan).values
ax_inset = fig.add_axes([0.1, 0.95, 0.7, 0.05], sharex=ax)
ax_inset.plot(time_array, shear_mag_series, lw=1.2)
ax_inset.set_ylabel("Shear (kt)", fontsize=8)
ax_inset.grid(True, ls=":", lw=0.5)
ax_inset.xaxis.set_visible(False)
ax_inset.set_ylim(np.nanmin(shear_mag_series) - 2, np.nanmax(shear_mag_series) + 2)

plt.subplots_adjust(bottom=0.12, top=0.9, left=0.1, right=0.95, hspace=0.3)
plt.show()


In [ ]:
# --- FIG. 5-style TB vs. instantaneous shear-relative azimuth time series ---

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from matplotlib.dates import DateFormatter
import matplotlib.dates as mdates
from tqdm import tqdm

# ---------------- PARAMETERS ----------------
inner_radius_deg = 3.0
azimuth_res_deg  = 3.0

time_start = pd.Timestamp("2014-09-11 12:00")
time_end   = pd.Timestamp("2014-09-18 12:00")

# ---------------- SELECT TIMES ----------------
times = track_interp.index[
    (track_interp.index >= time_start) &
    (track_interp.index <= time_end)
]

if len(times) == 0:
    raise ValueError("No times in track_interp within requested range.")

print(f"Processing {len(times)} time steps for full week.")

# ---------------- BUILD SHEAR FRAME ----------------
df_ships["shear_dir"] = (
    (np.degrees(np.arctan2(-df_ships["u_shear"], -df_ships["v_shear"])) + 360) % 360
)

df_ships_interp = df_ships.reindex(times).interpolate()

# ---------------- AZIMUTH BINS ----------------
az_edges = np.arange(0, 360 + azimuth_res_deg, azimuth_res_deg)
az_centers = (az_edges[:-1] + az_edges[1:]) / 2.0

# ---------------- SPATIAL SUBSET ----------------
lat_min, lat_max = track_interp["lat"].min() - 5, track_interp["lat"].max() + 5
lon_min, lon_max = track_interp["lon"].min() - 5, track_interp["lon"].max() + 5

ds_merg_small = ds_merg.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max)
)

# ---------------- PRECOMPUTE AZ / R GRIDS ----------------
lons, lats = ds_merg_small["lon"].values, ds_merg_small["lat"].values
lon2d, lat2d = np.meshgrid(lons, lats)

az_all, r_all = {}, {}

for storm_time in tqdm(times, desc="Precomputing az/r grids"):
    lat = float(track_interp.loc[storm_time, "lat"])
    lon = float(track_interp.loc[storm_time, "lon"])

    lat_rad = np.deg2rad(lat)
    dx = (lon2d - lon) * np.cos(lat_rad)
    dy = (lat2d - lat)

    az_all[storm_time] = (np.degrees(np.arctan2(dx, dy)) + 360) % 360
    r_all[storm_time] = np.sqrt(dx**2 + dy**2)

# ---------------- CONTAINER ----------------
tb_time_az = np.full((len(times), len(az_centers)), np.nan)

# ---------------- MAIN LOOP ----------------
for ti, storm_time in enumerate(tqdm(times, desc="Computing TB azimuth means")):

    tb = ds_merg_small["Tb"].sel(time=storm_time, method="nearest").values
    az_grid = az_all[storm_time]
    r_grid  = r_all[storm_time]

    # --- instantaneous shear direction ---
    shear_heading = df_ships_interp.loc[storm_time, "shear_dir"]

    # --- shear-relative azimuth ---
    rel_az = (az_grid - shear_heading) % 360

    # --- inner-core mask ---
    mask_inner = r_grid <= inner_radius_deg
    tb_vals = tb[mask_inner]

    # --- bin by azimuth ---
    az_bin_idx = np.digitize(rel_az[mask_inner], az_edges) - 1

    for ai in range(len(az_centers)):
        vals = tb_vals[az_bin_idx == ai]
        if vals.size > 0:
            tb_time_az[ti, ai] = np.nanmean(vals)

# ---------------- ROTATE SO 90° IS TOP ----------------
shift_bins = int(90 / azimuth_res_deg)
tb_time_az_rot = np.roll(tb_time_az, -shift_bins, axis=1)

az_edges_rot = np.arange(90, 90 + 360 + azimuth_res_deg, azimuth_res_deg)

# ---------------- TIME AXIS ----------------
time_array = np.array(times)

time_edges = pd.date_range(
    start=time_start,
    end=time_end + pd.Timedelta(minutes=30),
    freq="30min"
)
time_edges_num = mdates.date2num(time_edges)

# ---------------- PLOTTING ----------------
fig, ax = plt.subplots(figsize=(12, 5))

pcm = ax.pcolormesh(
    time_edges_num,
    az_edges_rot,
    tb_time_az_rot.T,
    cmap="turbo",
    vmin=180,
    vmax=300,
    shading="auto"
)

cbar = fig.colorbar(pcm, ax=ax, pad=0.02, label="Brightness Temperature (K)")
cbar.ax.invert_yaxis()

# ---------------- AXES ----------------
ax.set_ylabel("Azimuth relative to instantaneous shear (°)\n(0° = downshear)")
ax.set_ylim(90, 90 + 360)
ax.invert_yaxis()

yticks_pos = np.arange(90, 90 + 360, 45)
yticks_labels = (yticks_pos % 360).astype(int)
ax.set_yticks(yticks_pos)
ax.set_yticklabels(yticks_labels)

ax.set_xlim(time_start, time_end)

ax.set_title(
    f"Inner-3° TB vs Instantaneous Shear-Relative Azimuth\n"
    f"{time_start:%Y-%m-%d %H:%M} – {time_end:%Y-%m-%d %H:%M}",
    y=1.15
)

ax.xaxis_date()
ax.xaxis.set_major_formatter(DateFormatter('%m-%d\n%H:%M'))

# ---------------- DAILY LINES ----------------
for day in pd.date_range(
    time_start.normalize(),
    time_end.normalize() + pd.Timedelta(days=1),
    freq="D"
):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.6)

# ---------------- SHEAR MAG INSET ----------------
shear_mag_series = df_ships_interp["shear_mag"].values

ax_inset = fig.add_axes([0.1, 0.95, 0.7, 0.05], sharex=ax)
ax_inset.plot(time_array, shear_mag_series, lw=1.2)
ax_inset.set_ylabel("Shear (kt)", fontsize=8)
ax_inset.grid(True, ls=":", lw=0.5)
ax_inset.xaxis.set_visible(False)

ax_inset.set_ylim(
    np.nanmin(shear_mag_series) - 2,
    np.nanmax(shear_mag_series) + 2
)

plt.subplots_adjust(bottom=0.12, top=0.9, left=0.1, right=0.95)
fig.savefig("tb_vs_instant_shear_azimuth.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
print(len(az_centers), "azimuth bins")
print(tb_time_az.shape, "TB array shape")


In [ ]:
# Downshear asymmetry index

In [ ]:
# ---------------- DOWNSHEAR ASYMMETRY INDEX ----------------

dsai = np.full(len(times), np.nan)

for ti in range(len(times)):

    tb_slice = tb_time_az[ti, :]  # shape: (azimuth bins,)

    # skip if empty
    if np.all(np.isnan(tb_slice)):
        continue

    # downshear (0–180°)
    downshear_mask = (az_centers >= 0) & (az_centers < 180)
    upshear_mask   = (az_centers >= 180) & (az_centers < 360)

    tb_down = np.nanmean(tb_slice[downshear_mask])
    tb_up   = np.nanmean(tb_slice[upshear_mask])

    tb_all  = np.nanmean(tb_slice)

    dsai[ti] = (tb_up - tb_down) / tb_all

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

ax.plot(time_array, dsai, lw=1.5)

ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("DSAI")
ax.set_title("Downshear Asymmetry Index (TB)")
ax.grid(True, ls=":")

plt.show()

In [ ]:
# --- FIG. 5-style TB vs. instantaneous shear-relative azimuth time series ---

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr
from matplotlib.dates import DateFormatter
import matplotlib.dates as mdates
from tqdm import tqdm

# ============================================================
# PARAMETERS
# ============================================================
INNER_RADIUS_DEG     = 3.0    # degrees lat/lon (~330 km at 15°N)
AZIMUTH_RES_DEG      = 3.0    # azimuthal bin width
VMIN_TB              = 180    # colorbar min (coldest / deepest convection)
VMAX_TB              = 300    # colorbar max (warmest / suppressed)
SHEAR_MAG_THRESHOLD  = 5.0    # kt — below this, shear direction is unreliable

TIME_START = pd.Timestamp("2014-09-12 00:00")
TIME_END   = pd.Timestamp("2014-09-15 00:00")

# ============================================================
# SELECT TIMES FROM TRACK
# ============================================================
times = track_interp.index[
    (track_interp.index >= TIME_START) &
    (track_interp.index <= TIME_END)
]

if len(times) == 0:
    raise ValueError("No times found in track_interp within the requested range.")

print(f"Processing {len(times)} time steps for full week.")

# ============================================================
# SHEAR DIRECTION FROM SHIPS
# Convention: direction the shear vector points TOWARD
# (i.e., "downshear heading"), using standard met vector components.
# arctan2(-u, -v) gives the direction the wind is blowing FROM,
# so we negate both to get the direction it blows TOWARD.
# Double-check: if u_shear > 0 (westerly shear), shear_dir ~ 90° (eastward).
# ============================================================
df_ships["shear_dir"] = (
    (np.degrees(np.arctan2(-df_ships["u_shear"], -df_ships["v_shear"])) + 360) % 360
)

# Interpolate SHIPS onto track times
df_ships_interp = df_ships.reindex(
    df_ships.index.union(times)
).interpolate(method="time").reindex(times)

# --- Sanity check: no NaNs in shear after interpolation ---
nan_shear = df_ships_interp["shear_dir"].isna().sum()
nan_mag   = df_ships_interp["shear_mag"].isna().sum()
if nan_shear > 0 or nan_mag > 0:
    raise ValueError(
        f"NaN values found after interpolation: "
        f"{nan_shear} in shear_dir, {nan_mag} in shear_mag. "
        f"Check that SHIPS data covers {TIME_START} to {TIME_END}."
    )

# ============================================================
# MASK LOW-SHEAR TIMESTEPS
# When shear magnitude is near zero, shear direction is
# essentially noise from arctan2 of a near-zero vector.
# Set those rows to NaN so they don't pollute the plot.
# ============================================================
low_shear_mask = df_ships_interp["shear_mag"] < SHEAR_MAG_THRESHOLD
n_masked = low_shear_mask.sum()
if n_masked > 0:
    print(f"Warning: {n_masked} timesteps have shear mag < {SHEAR_MAG_THRESHOLD} kt "
          f"— TB azimuth means will be set to NaN for these times.")

# ============================================================
# AZIMUTH BINS
# ============================================================
az_edges   = np.arange(0, 360 + AZIMUTH_RES_DEG, AZIMUTH_RES_DEG)
az_centers = (az_edges[:-1] + az_edges[1:]) / 2.0
n_az       = len(az_centers)

# ============================================================
# SPATIAL SUBSET OF MERGIR (reduce memory / compute)
# ============================================================
PAD = 5.0
lat_min = track_interp["lat"].min() - PAD
lat_max = track_interp["lat"].max() + PAD
lon_min = track_interp["lon"].min() - PAD
lon_max = track_interp["lon"].max() + PAD

ds_merg_small = ds_merg.sel(
    lat=slice(lat_min, lat_max),
    lon=slice(lon_min, lon_max),
)

lons_grid = ds_merg_small["lon"].values
lats_grid = ds_merg_small["lat"].values
lon2d, lat2d = np.meshgrid(lons_grid, lats_grid)

# ============================================================
# PRECOMPUTE AZIMUTH AND RADIUS GRIDS FOR EACH TIME STEP
#
# Distance is computed in degrees (equirectangular):
#   dx = (lon - lon_center) * cos(lat_center_rad)  [scaled degrees]
#   dy = (lat - lat_center)                         [degrees]
#   r  = sqrt(dx^2 + dy^2)                         [degrees; ~111 km/deg]
#
# Consistent with INNER_RADIUS_DEG in the same units.
# Azimuth uses projection-corrected dx so angles are accurate.
# Valid approximation within ~5° of center at tropical latitudes.
# ============================================================
az_all = {}
r_all  = {}

for storm_time in tqdm(times, desc="Precomputing az/r grids"):
    lat_c   = float(track_interp.loc[storm_time, "lat"])
    lon_c   = float(track_interp.loc[storm_time, "lon"])
    cos_lat = np.cos(np.deg2rad(lat_c))

    dx = (lon2d - lon_c) * cos_lat
    dy = (lat2d - lat_c)

    az_all[storm_time] = (np.degrees(np.arctan2(dx, dy)) + 360) % 360
    r_all[storm_time]  = np.sqrt(dx**2 + dy**2)

# ============================================================
# MAIN LOOP: BIN TB BY SHEAR-RELATIVE AZIMUTH
# ============================================================
tb_time_az = np.full((len(times), n_az), np.nan)

for ti, storm_time in enumerate(tqdm(times, desc="Computing TB azimuth means")):

    # Skip low-shear times — direction is unreliable
    if low_shear_mask.loc[storm_time]:
        continue

    tb      = ds_merg_small["Tb"].sel(time=storm_time, method="nearest").values
    az_grid = az_all[storm_time]
    r_grid  = r_all[storm_time]

    shear_heading = df_ships_interp.loc[storm_time, "shear_dir"]
    rel_az        = (az_grid - shear_heading) % 360

    mask    = r_grid <= INNER_RADIUS_DEG
    tb_vals = tb[mask]
    az_vals = rel_az[mask]

    bin_idx = np.clip(np.digitize(az_vals, az_edges) - 1, 0, n_az - 1)

    for ai in range(n_az):
        sel = tb_vals[bin_idx == ai]
        if sel.size > 0:
            tb_time_az[ti, ai] = np.nanmean(sel)

# ============================================================
# ROTATE ARRAY SO PLOT Y-AXIS STARTS AT 90°
# ============================================================
shift_bins    = int(90 / AZIMUTH_RES_DEG)
tb_plot       = np.roll(tb_time_az, -shift_bins, axis=1)
az_edges_plot = np.arange(90, 90 + 360 + AZIMUTH_RES_DEG, AZIMUTH_RES_DEG)

# ============================================================
# TIME EDGES FOR PCOLORMESH (use actual data spacing)
# ============================================================
time_num = mdates.date2num(pd.DatetimeIndex(times))

dt_half_first  = (time_num[1]  - time_num[0])  / 2.0
dt_half_last   = (time_num[-1] - time_num[-2]) / 2.0
time_edges_num = np.concatenate([
    [time_num[0]  - dt_half_first],
    (time_num[:-1] + time_num[1:]) / 2.0,
    [time_num[-1] + dt_half_last],
])

# ============================================================
# PLOTTING
# ============================================================
fig = plt.figure(figsize=(14, 6))
ax  = fig.add_axes([0.08, 0.10, 0.75, 0.72])

pcm = ax.pcolormesh(
    time_edges_num,
    az_edges_plot,
    tb_plot.T,
    cmap="turbo",           # standard turbo: warm = cold TB (deep convection)
    vmin=VMIN_TB,
    vmax=VMAX_TB,
    shading="flat",
)

# Colorbar — inverted so cold (deep convection) is at top
cbar = fig.colorbar(pcm, ax=ax, pad=0.02)
cbar.set_label("Brightness Temperature (K)", fontsize=10)
cbar.ax.invert_yaxis()      # cold TB at top, warm at bottom

# Y-axis
ax.set_ylabel("Azimuth relative to instantaneous shear (°)\n(0° = downshear)", fontsize=10)
ax.set_ylim(90, 90 + 360)
ax.invert_yaxis()

ytick_pos    = np.arange(90, 90 + 360, 45)
ytick_labels = [str(int(v % 360)) for v in ytick_pos]
ax.set_yticks(ytick_pos)
ax.set_yticklabels(ytick_labels)

# Quadrant boundary lines
for boundary in np.arange(90, 90 + 360 + 45, 45):
    ax.axhline(boundary, color="white", lw=0.5, ls="--", alpha=0.4)

# Quadrant labels — placed on LEFT side in axes-fraction coords
quadrant_info = [
    (90 + 22.5,  "DSR"),
    (90 + 67.5,  "DSL"),
    (90 + 112.5, "USL"),
    (90 + 157.5, "USR"),
]
y_data_min, y_data_max = 90, 90 + 360

for y_data, label in quadrant_info:
    y_frac = 1.0 - (y_data - y_data_min) / (y_data_max - y_data_min)
    ax.text(
        -0.01, y_frac, label,
        fontsize=7, va="center", ha="right", color="white",
        fontweight="bold", transform=ax.transAxes,
    )

# X-axis
ax.set_xlim(time_edges_num[0], time_edges_num[-1])
ax.xaxis_date()
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(DateFormatter('%m-%d\n%H:%M'))

# Daily vertical lines
for day in pd.date_range(
    TIME_START.normalize(),
    TIME_END.normalize() + pd.Timedelta(days=1),
    freq="D"
):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.5)

ax.set_title(
    f"Inner-{INNER_RADIUS_DEG:.0f}° TB vs Instantaneous Shear-Relative Azimuth  "
    f"(MergIR)\n{TIME_START:%Y-%m-%d %H:%M} – {TIME_END:%Y-%m-%d %H:%M}",
    fontsize=11, pad=8
)

# ============================================================
# SHEAR MAGNITUDE INSET
# ============================================================
ax_shear = fig.add_axes([0.08, 0.84, 0.75, 0.10], sharex=ax)

shear_mag = df_ships_interp["shear_mag"].values
ax_shear.plot(time_num, shear_mag, lw=1.4, color="steelblue")
ax_shear.fill_between(time_num, shear_mag, alpha=0.15, color="steelblue")

# Shade low-shear masked region in the inset
ax_shear.axhspan(0, SHEAR_MAG_THRESHOLD, color="red", alpha=0.08, zorder=0)
ax_shear.axhline(SHEAR_MAG_THRESHOLD, color="red", lw=0.8, ls="--", alpha=0.5,
                 label=f"< {SHEAR_MAG_THRESHOLD} kt threshold")
ax_shear.legend(fontsize=6, loc="upper left")

ax_shear.set_ylabel("Shear\n(kt)", fontsize=8)
ax_shear.grid(True, ls=":", lw=0.5, alpha=0.6)
ax_shear.xaxis.set_visible(False)
ax_shear.set_ylim(0, np.nanmax(shear_mag) + 3)
ax_shear.set_xlim(time_edges_num[0], time_edges_num[-1])

fig.savefig("tb_vs_instant_shear_azimuth.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# PLOTTING (REWRITTEN CLEAN VERSION)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter
import matplotlib.patheffects as pe

# --- Figure layout using GridSpec (cleaner than add_axes) ---
fig = plt.figure(figsize=(14, 6))
gs = fig.add_gridspec(
    nrows=2, ncols=1,
    height_ratios=[1, 8],   # thinner shear panel
    hspace=0.05
)

ax_shear = fig.add_subplot(gs[0])
ax       = fig.add_subplot(gs[1], sharex=ax_shear)

# ============================================================
# MAIN TB PANEL
# ============================================================
pcm = ax.pcolormesh(
    time_edges_num,
    az_edges_plot,
    tb_plot.T,
    cmap="turbo",
    vmin=VMIN_TB,
    vmax=VMAX_TB,
    shading="flat",
)

# Colorbar
cbar = fig.colorbar(pcm, ax=ax, pad=0.02)
cbar.set_label("Brightness Temperature (K)", fontsize=10)
cbar.ax.invert_yaxis()

# Y-axis formatting
ax.set_ylabel(
    "Azimuth relative to instantaneous shear (°)\n(0° = downshear)",
    fontsize=10
)
ax.set_ylim(90, 90 + 360)
ax.invert_yaxis()

ytick_pos = np.arange(90, 90 + 360, 45)
ax.set_yticks(ytick_pos)
ax.set_yticklabels([str(int(v % 360)) for v in ytick_pos])

# Quadrant boundary lines
for boundary in np.arange(90, 90 + 360 + 45, 45):
    ax.axhline(boundary, color="white", lw=0.5, ls="--", alpha=0.4)


# ============================================================
# X-axis formatting
# ============================================================
ax.set_xlim(time_edges_num[0], time_edges_num[-1])
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(DateFormatter('%m-%d\n%H:%M'))

# Daily vertical lines
for day in pd.date_range(
    TIME_START.normalize(),
    TIME_END.normalize() + pd.Timedelta(days=1),
    freq="D"
):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.5)

# Title
ax.set_title(
    f"Inner-{INNER_RADIUS_DEG:.0f}° TB vs Instantaneous Shear-Relative Azimuth (MergIR)\n"
    f"{TIME_START:%Y-%m-%d %H:%M} – {TIME_END:%Y-%m-%d %H:%M}",
    fontsize=11,
    pad=50
)

# ============================================================
# SHEAR PANEL (THINNER + CLEANER)
# ============================================================
shear_mag = df_ships_interp["shear_mag"].values

ax_shear.plot(time_num, shear_mag, lw=1)
ax_shear.fill_between(time_num, shear_mag, alpha=0.2)

# Threshold shading
ax_shear.axhspan(0, SHEAR_MAG_THRESHOLD, alpha=0.1)


ax_shear.set_ylabel("Shear (kt)", fontsize=8)
ax_shear.set_ylim(0, np.nanmax(shear_mag) + 5)
ax_shear.grid(True, ls=":", lw=0.5, alpha=0.6)

# Hide x labels on top panel
ax_shear.tick_params(labelbottom=False)

# ============================================================
# SAVE + SHOW
# ============================================================
fig.savefig(
    "tb_vs_instant_shear_azimuth.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
def debug_shear_alignment_once(storm_time, ds_merg_small, az_all, df_ships_interp):
    """
    One-time, guaranteed debug for shear-relative azimuth correctness.
    """

    print("\n" + "="*70)
    print("DEBUG FUNCTION CALLED")
    print("="*70)

    # -------------------------------
    # Pull data for one timestep
    # -------------------------------
    tb = ds_merg_small["Tb"].sel(time=storm_time, method="nearest").values
    az_grid = az_all[storm_time]

    u = df_ships_interp.loc[storm_time, "u_shear"]
    v = df_ships_interp.loc[storm_time, "v_shear"]

    print(f"\nTime: {storm_time}")
    print(f"u_shear: {u:.2f}, v_shear: {v:.2f}")

    # -------------------------------
    # Compute shear directions
    # -------------------------------
    shear_from = (np.degrees(np.arctan2(u, v)) + 360) % 360
    shear_to   = (shear_from + 180) % 360
    shear_alt  = (np.degrees(np.arctan2(-u, -v)) + 360) % 360

    print("\nShear direction definitions:")
    print(f"FROM: {shear_from:.2f}")
    print(f"TO:   {shear_to:.2f}")
    print(f"ALT:  {shear_alt:.2f}")

    # -------------------------------
    # Find coldest TB location
    # -------------------------------
    tb_flat = tb.flatten()
    az_flat = az_grid.flatten()

    valid = ~np.isnan(tb_flat)
    tb_flat = tb_flat[valid]
    az_flat = az_flat[valid]

    cold_idx = np.argmin(tb_flat)
    cold_tb  = tb_flat[cold_idx]
    cold_az  = az_flat[cold_idx]

    print("\nColdest convection signal:")
    print(f"TB: {cold_tb:.2f} K")
    print(f"Azimuth: {cold_az:.2f}°")

    # -------------------------------
    # Angular difference function
    # -------------------------------
    def ang_diff(a, b):
        return np.abs((a - b + 180) % 360 - 180)

    diff_from = ang_diff(cold_az, shear_from)
    diff_to   = ang_diff(cold_az, shear_to)
    diff_alt  = ang_diff(cold_az, shear_alt)

    print("\nAlignment with shear:")
    print(f"Offset from FROM: {diff_from:.2f}°")
    print(f"Offset from TO:   {diff_to:.2f}°")
    print(f"Offset from ALT:  {diff_alt:.2f}°")

    # -------------------------------
    # Determine best match
    # -------------------------------
    diffs = {
        "FROM": diff_from,
        "TO": diff_to,
        "ALT": diff_alt
    }

    best = min(diffs, key=diffs.get)

    print("\nINTERPRETATION:")
    if best == "TO":
        print("✅ Convection aligns with shear TO direction (correct physics)")
    elif best == "FROM":
        print("⚠️ Convection aligns with shear FROM direction → you need to flip by 180°")
    else:
        print("⚠️ Your (-u, -v) method is closest → likely inconsistent convention")

    print("="*70 + "\n")

In [ ]:
test_time = times[0]  # or any time you want

debug_shear_alignment_once(
    test_time,
    ds_merg_small,
    az_all,
    df_ships_interp
)

In [ ]:
# ============================================================
# FIXED AZIMUTH CALCULATION (CRITICAL FIX)
# ============================================================
az_all = {}
r_all  = {}

for storm_time in times:
    lat_c   = float(track_interp.loc[storm_time, "lat"])
    lon_c   = float(track_interp.loc[storm_time, "lon"])
    cos_lat = np.cos(np.deg2rad(lat_c))

    dx = (lon2d - lon_c) * cos_lat
    dy = (lat2d - lat_c)

    # ✅ FIX: correct orientation
    az_all[storm_time] = (np.degrees(np.arctan2(dy, dx)) + 360) % 360
    r_all[storm_time]  = np.sqrt(dx**2 + dy**2)


# ============================================================
# MAIN LOOP (unchanged logic, now correct physics)
# ============================================================
tb_time_az = np.full((len(times), len(az_centers)), np.nan)

for ti, storm_time in enumerate(times):

    tb      = ds_merg_small["Tb"].sel(time=storm_time, method="nearest").values
    az_grid = az_all[storm_time]
    r_grid  = r_all[storm_time]

    shear_heading = df_ships_interp.loc[storm_time, "shear_dir"]

    # still correct
    rel_az = (az_grid - shear_heading) % 360

    mask    = r_grid <= INNER_RADIUS_DEG
    tb_vals = tb[mask]
    az_vals = rel_az[mask]

    bin_idx = np.clip(np.digitize(az_vals, az_edges) - 1, 0, len(az_centers) - 1)

    for ai in range(len(az_centers)):
        vals = tb_vals[bin_idx == ai]
        if vals.size > 0:
            tb_time_az[ti, ai] = np.nanmean(vals)


# ============================================================
# ROTATE SO 0° (DOWNSHEAR) IS CENTERED PROPERLY
# ============================================================
shift_bins = int(90 / AZIMUTH_RES_DEG)
tb_plot    = np.roll(tb_time_az, -shift_bins, axis=1)

az_edges_plot = np.arange(90, 90 + 360 + AZIMUTH_RES_DEG, AZIMUTH_RES_DEG)


# ============================================================
# TIME AXIS (MATCH DATA — FIXED)
# ============================================================
time_array = pd.DatetimeIndex(times)
time_num   = mdates.date2num(time_array)

dt = np.diff(time_num)
dt = np.append(dt, dt[-1])

time_edges_num = np.concatenate([
    [time_num[0] - dt[0] / 2],
    time_num[:-1] + dt[:-1] / 2,
    [time_num[-1] + dt[-1] / 2],
])


# ============================================================
# PLOTTING
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter

fig = plt.figure(figsize=(14, 6))

gs = fig.add_gridspec(
    nrows=2, ncols=1,
    height_ratios=[1, 7],
    hspace=0.04
)

ax_shear = fig.add_subplot(gs[0])
ax       = fig.add_subplot(gs[1], sharex=ax_shear)

# ---------------- MAIN PANEL ----------------
pcm = ax.pcolormesh(
    time_edges_num,
    az_edges_plot,
    tb_plot.T,
    cmap="turbo",
    vmin=VMIN_TB,
    vmax=VMAX_TB,
    shading="flat"
)

cbar = fig.colorbar(pcm, ax=ax, pad=0.02)
cbar.set_label("Brightness Temperature (K)", fontsize=10)
cbar.ax.invert_yaxis()

# Y-axis
ax.set_ylabel(
    "Azimuth relative to instantaneous shear (°)\n(0° = downshear)",
    fontsize=10
)

ax.set_ylim(90, 90 + 360)
ax.invert_yaxis()

yticks = np.arange(90, 90 + 360, 45)
ax.set_yticks(yticks)
ax.set_yticklabels([str(int(v % 360)) for v in yticks])

# quadrant lines
for y in np.arange(90, 90 + 360 + 45, 45):
    ax.axhline(y, color="white", lw=0.5, ls="--", alpha=0.4)

# X-axis
ax.set_xlim(time_edges_num[0], time_edges_num[-1])
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(DateFormatter('%m-%d\n%H:%M'))

# daily lines
for day in pd.date_range(
    TIME_START.normalize(),
    TIME_END.normalize() + pd.Timedelta(days=1),
    freq="D"
):
    ax.axvline(mdates.date2num(day), color="white", lw=0.6, alpha=0.5)

# title
ax.set_title(
    f"Inner-{INNER_RADIUS_DEG:.0f}° TB vs Instantaneous Shear-Relative Azimuth (MergIR)\n"
    f"{TIME_START:%Y-%m-%d %H:%M} – {TIME_END:%Y-%m-%d %H:%M}",
    fontsize=11,
    pad=12
)

# ---------------- SHEAR PANEL ----------------
shear_mag = df_ships_interp["shear_mag"].values

ax_shear.plot(time_num, shear_mag, lw=1)
ax_shear.fill_between(time_num, shear_mag, alpha=0.2)

ax_shear.axhspan(0, SHEAR_MAG_THRESHOLD, alpha=0.1)

ax_shear.set_ylabel("Shear (kt)", fontsize=8)
ax_shear.set_ylim(0, np.nanmax(shear_mag) + 5)
ax_shear.grid(True, ls=":", lw=0.5, alpha=0.6)

ax_shear.tick_params(labelbottom=False)

# ---------------- SAVE ----------------
fig.savefig(
    "tb_vs_instant_shear_azimuth.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()